In [ ]:
!pip install transformers torch -q

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

Im testing tokenization and next word prediction on GPT2.
1. we have imported both functions for tokenization and next word prediction.
2. using eval function for inference.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

model.eval()

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

Testing tokenization on different edge cases.
1. how lettercase effects tokenization.
2. punctuation creates extra tokens and splits a single word into two.
3. how sequence of number gets treated in tokenization
4. how whitespace also creates extra tokens
5. encoding each token ID for positional awareness.

In [ ]:
texts = [
    "Hello",
    "hello",
    "Hello world",
    "HelloWorld",
    "hello_world",
    "123456",
    "AI!!!",
    "don't",
    "ChatGPT",
    "        rida"
]

for text in texts:
    print(text, "→", tokenizer.tokenize(text))
    ids = tokenizer.encode(text)
    print(ids)

Hello → ['Hello']
[15496]
hello → ['hello']
[31373]
Hello world → ['Hello', 'Ġworld']
[15496, 995]
HelloWorld → ['Hello', 'World']
[15496, 10603]
hello_world → ['hello', '_', 'world']
[31373, 62, 6894]
123456 → ['123', '456']
[10163, 29228]
AI!!! → ['AI', '!!!']
[20185, 10185]
don't → ['don', "'t"]
[9099, 470]
ChatGPT → ['Chat', 'G', 'PT']
[30820, 38, 11571]
        rida → ['Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġ', 'Ġr', 'ida']
[220, 220, 220, 220, 220, 220, 220, 374, 3755]


This function is the logic for next word prediction.
1. first the input is getting tokenized
2. tokens are passed to model for next word prediction.
3. from the output of final layer we are only going to take the last logit
4. converting those logits into probabilities that add upto 1
5. only top 5 most probable words are considered as the final possible word
6. decoding tokens into readible form

In [ ]:
def next_token_predictions(prompt, top_k=5):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    # Get logits for the last token
    logits = outputs.logits[:, -1, :]

    # Convert logits to probabilities
    probs = torch.softmax(logits, dim=-1)

    # Get top-k tokens
    top_probs, top_ids = torch.topk(probs, top_k)

    print(f"\nPrompt: {prompt}")
    print("-" * 40)

    for prob, token_id in zip(top_probs[0], top_ids[0]):
        token = tokenizer.decode([token_id])
        print(f"{repr(token):15} {prob.item():.4f}")

Trying next word prediction on different sentences.

In [ ]:
prompts = [
    "The capital of France is",
    "The capital of Pakistan is",
    "I like to eat",
    "Once upon a time",
    "The cat sat on the"
]

for prompt in prompts:
    next_token_predictions(prompt)


Prompt: The capital of France is
----------------------------------------
' the'          0.0846
' now'          0.0479
' a'            0.0462
' France'       0.0324
' Paris'        0.0322

Prompt: The capital of Pakistan is
----------------------------------------
' the'          0.1004
' a'            0.0552
' Islamabad'    0.0387
' home'         0.0347
' now'          0.0293

Prompt: I like to eat
----------------------------------------
' a'            0.0638
','             0.0372
' and'          0.0340
' my'           0.0333
'.'             0.0255

Prompt: Once upon a time
----------------------------------------
','             0.4269
' the'          0.0646
' I'            0.0406
' he'           0.0369
' there'        0.0294

Prompt: The cat sat on the
----------------------------------------
' floor'        0.0764
' bed'          0.0653
' couch'        0.0541
' ground'       0.0521
' edge'         0.0478


Trying the same sentence with different context.

In [ ]:
next_token_predictions("The cat sat on the")
next_token_predictions("The cat sat on the mat and")


Prompt: The cat sat on the
----------------------------------------
' floor'        0.0764
' bed'          0.0653
' couch'        0.0541
' ground'       0.0521
' edge'         0.0478

Prompt: The cat sat on the mat and
----------------------------------------
' looked'       0.0536
' watched'      0.0394
' stared'       0.0349
' the'          0.0196
' was'          0.0174


Trying on the same word with different semantics and context.

In [ ]:
next_token_predictions("I went to the bank to deposit")
next_token_predictions("I went to the river bank and sat")


Prompt: I went to the bank to deposit
----------------------------------------
' my'           0.2256
' the'          0.1420
' money'        0.1166
' it'           0.0626
' $'            0.0519

Prompt: I went to the river bank and sat
----------------------------------------
' down'         0.6197
' on'           0.1048
' in'           0.0754
' there'        0.0311
' with'         0.0203
